Auto Loader discovers new files and uses a checkpoint to remember ingestion progress. It is built on Spark Structured Streaming and supports CSV, JSON, Parquet, and other formats



flowchart TD
    A["📁 Incoming CSV files<br/>orders_batch_001.csv, orders_batch_002.csv, ..."]
    B["🔄 Auto Loader<br/>Discovers files not yet processed"]
    C["🗄️ Bronze Delta table<br/>Stores incoming records"]
    D["🔖 Checkpoint<br/>Remembers progress"]

    A --> B
    B --> C
    B --> D

    style A fill:#f8f9fa,stroke:#dddddd,color:#111111
    style B fill:#f8f9fa,stroke:#dddddd,color:#111111
    style C fill:#f8f9fa,stroke:#dddddd,color:#111111
    style D fill:#f8f9fa,stroke:#dddddd,color:#111111



Bronze layer is the raw landing layer in your lakehouse.

It stores source records with minimal transformations. We aren't trying to determine the latest version of each order yet.

For example, if an order appears in Monday's CSV and again in Tuesday's CSV with a new status, both versions can exist in Bronze. 

A checkpoint stores info that helps Spark and Auto Loader recover and resume processing

Without reliable progress tracking, restarting pipeline may cause previously processed data to be ingested again

Auto Loader tracks file discovery state in the checkpoint and supports exactly once processing with Delta Lake under its documented guarantees

In [ ]:
catalog = "workspace"

schema = "incremental_project"
volume = "source_files"

base_path = f"/Volumes/{catalog}/{schema}/{volume}"
#source_path purpose is where Auto Loader discovers CSV files
#schema_path purpose is where Auto Loader stores inferred schema information
#checkpoint_path purpose is where processing state is stored
#table_name purpose is where ingested records are written
source_path = f"{base_path}/incoming"
schema_path = f"{base_path}/_schemas/orders_v2"
checkpoint_path = f"{base_path}/_checkpoints/orders_v2"
table_name = f"{catalog}.{schema}.orders_bronze_v2"

print("Source:", source_path)
print("Schema location:", schema_path)
print("Checkpoint:", checkpoint_path)
print("Target table:", table_name)

In [ ]:
from pyspark.sql import functions as F
#spark.readStream - Creates a streaming DataFrame reader. Unlike spark.read, this sets up a streaming source that can process data incrementally
#.format("cloudFiles") - Selects Databricks Auto Loader. cloudFiles is name of its Structured Streaming Source
#.option("cloudFiles.format", "csv") - Specifies that the incoming files are CSVs
#.option("cloudFiles.schemaLocation", schema_path) - provides a location where Auto Loader can retain schema information. This helps it track the source structure and handle future schema changes
#.option("header", "true") - Tells Spark that the first line of each CSV contains column names
#.load(source_path) - Points Auto loader at our source folder
orders_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format","csv")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("header", "true")
    .load(source_path)
)

In [ ]:
incoming_path = f"{base_path}/incoming"
source_path = incoming_path

dbutils.fs.mkdirs(incoming_path)

original_file = f"{base_path}/orders_batch_001.csv"
incoming_file = f"{incoming_path}/orders_batch_001.csv"

if not any(
    f.name == "orders_batch_001.csv"
    for f in dbutils.fs.ls(incoming_path)
):
    dbutils.fs.cp(original_file, incoming_file)
display(dbutils.fs.ls(incoming_path))

In [ ]:
from pyspark.sql import functions as F

# Remove leading and trailing spaces from column names
orders_clean_columns = orders_stream.toDF(
    *[column.strip() for column in orders_stream.columns]
)
#withColumn("_ingested_at", F.current_timestamp()) adds a timestamp indicating when Spark processed the record. This is useful when troubleshooting ingestion
#withColumn("_source_file",F.col("_metadata.file_path")) records the originating CSV file, helping us trace records back to their source
bronze_stream = (
    orders_clean_columns
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)
#write Stream starts configuring output side of streaming pipeline
#.format("delta") specifies that the records will be stored in Delta Lake, which provides transactional reliability and supports later operations such as MERGE
#.option("checkpointLocation", checkpoint_path) tells Spark where to store processing state
#.trigger(availableNow=True) processes files available when this run begins, then stops instead of remaining active indefinitely
#.toTable(table_name) starts streaming write into our Bronze table
query = (
    bronze_stream
    .writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(table_name)
)
#query.awaitTermination() waits for this run to finish before the next notebook cell continues
query.awaitTermination()
print("Bronze ingestion completed")

In [ ]:
#Validation A - Check your first ingestion
bronze_df = spark.table(table_name)

display(bronze_df.orderBy("order_id"))

print("Bronze count:", bronze_df.count())

In [ ]:
display(
    bronze_df.groupBy("_source_file")
    .count()
)

In [ ]:
assert bronze_df.count() == 5, "Expected 5 Bronze records"

assert bronze_df.select("order_id").distinct().count() == 5

print("Validation A passed")

In [ ]:
count_after_rerun = spark.table(table_name).count()

print("Rows after rerun:", count_after_rerun)

assert count_after_rerun == 5

print("Validation B passed: No duplicate file ingestion")

Auto Loader finds and processes new source files
readStream defines an incremental data source
cloudFiles is Databricks Auto Loader source format
Schema location stores info about incoming data structure
Checkpoint stores processing progress for recovery
availableNow=True processes available data and stops
Bronze stores source records with minimal transformation
Delta Lake reliable table format for our lakehouse